# Clase 03 — Versionado de datos y modelos

**Curso:** MLOps y Monitorización con Python · **Bloque 2:** Versionar y medir
**Tiempo estimado:** 100 minutos · **Capítulo del manual:** `manual/03_versionado_datos_modelos.md`

## Qué vas a ver en esta clase

- Por qué **Git no basta** para versionar datos y modelos (archivos grandes, binarios, que cambian por completo).
- **DVC** (`dvc init`, `dvc add`) para versionar `datos/churn.csv`, con un **remoto local** (una carpeta, no la nube).
- El **linaje** de una versión: de qué código y de qué datos de origen sale.
- El principio de **datos inmutables**: nunca sobrescribir, siempre crear una versión nueva.
- Recuperar una versión anterior con `git checkout` + `dvc checkout`, y **verificarlo de verdad** leyendo el archivo.

Todo ocurre dentro de una carpeta de trabajo efímera (`proyecto_churn/`), creada y destruida por este mismo notebook: no tocamos el repositorio del curso.

## Requisitos

- `pandas` y `dvc` (la celda siguiente instala solo lo que falte, también en Google Colab). `git` debe estar disponible en el sistema (lo está en Colab y en la mayoría de instalaciones).
- La carpeta `datos/` del curso, con `churn.csv` dentro.

> **Cómo usar este notebook:** ejecuta las celdas en orden, de arriba a abajo, una sola vez.

---
## Instalación

Instala solo lo que falte.

In [ ]:
import subprocess, sys


def asegurar(paquete, modulo):
    """Instala el paquete solo si falta (idempotente y silencioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)


asegurar("numpy", "numpy")
asegurar("pandas", "pandas")
asegurar("dvc", "dvc")
print("Librerías listas.")

---
## Celda 0 — Setup

Definimos `cargar_datos` (para encontrar `churn.csv` en `datos/`, `../datos/` o `/content/datos/`) y dos funciones auxiliares: `ejecutar`, que lanza un comando de línea de órdenes con `subprocess.run(..., capture_output=True, text=True)`, y `mostrar`, que imprime su resultado de forma compacta. No hay ninguna fuente de aleatoriedad en este notebook, así que no hace falta fijar ninguna semilla.

In [ ]:
import os
import shutil
import subprocess
import sys

import pandas as pd

def cargar_datos(nombre):
    """Busca un archivo de datos en datos/, ../datos/ y /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/18-mlops-monitoraggio/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "No encuentro " + nombre + ". Sube la carpeta datos/ del curso (en Colab: panel de archivos) o déjala junto al notebook.")

def ejecutar(comando, cwd):
    """Ejecuta un comando de línea de órdenes y devuelve su resultado, sin lanzar excepción si falla."""
    return subprocess.run(comando, cwd=cwd, capture_output=True, text=True)


def mostrar(etiqueta, resultado):
    """Imprime el resultado de un comando de forma compacta, útil para seguir el notebook."""
    print(f"$ {etiqueta}  (rc={resultado.returncode})")
    salida = (resultado.stdout + resultado.stderr).strip()
    if salida:
        for linea in salida.splitlines()[:8]:
            print("   ", linea)

# Resolvemos la ruta del ejecutable "dvc": suele instalarse junto al propio intérprete de Python
DVC = shutil.which("dvc") or os.path.join(os.path.dirname(sys.executable), "dvc")

# Fecha de autor/commit fija: así el hash de cada commit es el mismo en cada ejecución (reproducible)
for variable in ("GIT_AUTHOR_DATE", "GIT_COMMITTER_DATE"):
    os.environ[variable] = "2024-01-01T00:00:00+00:00"

print("Librerías y funciones auxiliares listas.")
print("Ejecutable de dvc:", DVC)

---
## 1. Carpeta de trabajo efímera

Creamos `proyecto_churn/` (y, junto a ella, `proyecto_churn_remoto/`) desde cero en cada ejecución, y copiamos `churn.csv` dentro: el original de `datos/` no se toca.

In [ ]:
CARPETA = "proyecto_churn"
REMOTO = "proyecto_churn_remoto"

# Limpiamos y recreamos ambas carpetas para que el notebook sea repetible desde cero
for carpeta in (CARPETA, REMOTO):
    if os.path.exists(carpeta):
        shutil.rmtree(carpeta)
os.makedirs(CARPETA, exist_ok=True)

shutil.copyfile(cargar_datos("churn.csv"), os.path.join(CARPETA, "churn.csv"))
print("Carpeta de trabajo creada:", os.path.abspath(CARPETA))
print("churn.csv copiado dentro de la carpeta efímera (el original en datos/ no se toca).")

---
## 2. `git init` + `dvc init`

Inicializamos un repositorio Git de verdad, con una identidad local de prueba, y luego DVC.

In [ ]:
mostrar("git init", ejecutar(["git", "init"], cwd=CARPETA))
mostrar("git config user.email", ejecutar(["git", "config", "user.email", "curso@ejemplo.local"], cwd=CARPETA))
mostrar("git config user.name", ejecutar(["git", "config", "user.name", "Curso MLOps"], cwd=CARPETA))
mostrar("dvc init", ejecutar([DVC, "init"], cwd=CARPETA))

assert os.path.isdir(os.path.join(CARPETA, ".git"))
assert os.path.isdir(os.path.join(CARPETA, ".dvc"))
print(".git y .dvc inicializados dentro de", CARPETA)

---
## 3. Un remoto local de DVC

Registramos `../proyecto_churn_remoto` como remoto por defecto: una carpeta, no un servicio en la nube.

In [ ]:
ruta_remoto = os.path.join("..", REMOTO)
r = ejecutar([DVC, "remote", "add", "-d", "remoto_local", ruta_remoto], cwd=CARPETA)
mostrar("dvc remote add -d remoto_local " + ruta_remoto, r)
assert r.returncode == 0

r = ejecutar([DVC, "remote", "list"], cwd=CARPETA)
mostrar("dvc remote list", r)

---
## 4. Versionar la v1 de `churn.csv`

`dvc add` calcula el hash y crea `churn.csv.dvc`; lo confirmamos con Git y lo enviamos al remoto con `dvc push`.

In [ ]:
r = ejecutar([DVC, "add", "churn.csv"], cwd=CARPETA)
mostrar("dvc add churn.csv", r)
assert r.returncode == 0
assert os.path.exists(os.path.join(CARPETA, "churn.csv.dvc"))

with open(os.path.join(CARPETA, "churn.csv.dvc"), encoding="utf-8") as f:
    contenido_dvc_v1 = f.read()
print("Contenido de churn.csv.dvc (v1):")
print(contenido_dvc_v1)

mostrar("git add", ejecutar(["git", "add", "churn.csv.dvc", ".gitignore"], cwd=CARPETA))
mostrar("git commit v1", ejecutar(["git", "commit", "-m", "v1: churn.csv original"], cwd=CARPETA))
mostrar("dvc push", ejecutar([DVC, "push"], cwd=CARPETA))

r = ejecutar(["git", "rev-parse", "HEAD"], cwd=CARPETA)
hash_commit_v1 = r.stdout.strip()
print("Commit de la v1:", hash_commit_v1)

**¿Qué observas?** `churn.csv.dvc` es un archivo de texto pequeño (unas pocas líneas) con el hash MD5 y el tamaño de `churn.csv`: es justo lo que Git puede versionar bien. El archivo grande de verdad ya está en el remoto tras `dvc push`.

---
## 5. Una segunda limpieza real: la v2

Limitamos `tickets_soporte` a un máximo de 5 (unos pocos valores atípicos) y volvemos a versionar.

In [ ]:
df = pd.read_csv(os.path.join(CARPETA, "churn.csv"))
suma_antes = int(df["tickets_soporte"].sum())
maximo_antes = int(df["tickets_soporte"].max())
filas_afectadas = int((df["tickets_soporte"] > 5).sum())

# Segunda limpieza: limitamos los tickets de soporte a un máximo de 5 (outliers)
df["tickets_soporte"] = df["tickets_soporte"].clip(upper=5)
df.to_csv(os.path.join(CARPETA, "churn.csv"), index=False)

suma_despues = int(df["tickets_soporte"].sum())
maximo_despues = int(df["tickets_soporte"].max())

print("filas con más de 5 tickets (afectadas por el clip):", filas_afectadas)
print("máximo antes/después:", maximo_antes, "→", maximo_despues)
print("suma antes/después:", suma_antes, "→", suma_despues)
assert maximo_despues == 5 and maximo_antes > maximo_despues

In [ ]:
r = ejecutar([DVC, "add", "churn.csv"], cwd=CARPETA)
mostrar("dvc add churn.csv (v2)", r)
assert r.returncode == 0

with open(os.path.join(CARPETA, "churn.csv.dvc"), encoding="utf-8") as f:
    contenido_dvc_v2 = f.read()
print("Contenido de churn.csv.dvc (v2):")
print(contenido_dvc_v2)
assert contenido_dvc_v2 != contenido_dvc_v1, "el hash de la v2 debe ser distinto al de la v1"

mostrar("git add", ejecutar(["git", "add", "churn.csv.dvc"], cwd=CARPETA))
mostrar("git commit v2", ejecutar(["git", "commit", "-m", "v2: tickets_soporte limitado a 5 (outliers)"], cwd=CARPETA))
mostrar("dvc push", ejecutar([DVC, "push"], cwd=CARPETA))

**¿Qué observas?** El contenido de `churn.csv.dvc` cambió (hash distinto): es una versión nueva, no una actualización de la anterior. El `churn.csv.dvc` de la v1 sigue existiendo en el historial de Git (en el commit anterior), aunque en la carpeta de trabajo ya no lo veas.

---
## 6. El principio de datos inmutables, en espacio real

Contamos cuántos objetos distintos hay en el remoto: uno por cada versión, porque DVC nunca sobrescribe un hash ya guardado.

In [ ]:
# Contamos cuántos objetos distintos guarda el remoto: uno por cada versión, nunca se sobrescriben
objetos_remoto = []
for raiz, _, archivos in os.walk(REMOTO):
    for nombre in archivos:
        objetos_remoto.append(os.path.join(raiz, nombre))

peso_total_kb = sum(os.path.getsize(p) for p in objetos_remoto) / 1024
print("objetos guardados en el remoto (uno por versión distinta):", len(objetos_remoto))
print("espacio total ocupado en el remoto (KB):", round(peso_total_kb, 1))
assert len(objetos_remoto) == 2, "deberían existir exactamente 2 versiones guardadas (v1 y v2)"

**¿Qué observas?** Dos versiones de un archivo de ≈150 KB ocupan en el remoto casi el doble de espacio que una sola: es el precio de la inmutabilidad. Con datasets grandes y muchas versiones, este coste hay que tenerlo en cuenta (por ejemplo, limpiando versiones muy antiguas del remoto cuando ya nadie las necesita).

---
## 7. Recuperar la v1 con `git checkout` + `dvc checkout`

Primero recuperamos el puntero (`.dvc`) del commit de la v1, luego sincronizamos el archivo real.

In [ ]:
r = ejecutar(["git", "checkout", hash_commit_v1, "--", "churn.csv.dvc"], cwd=CARPETA)
mostrar("git checkout " + hash_commit_v1[:8] + " -- churn.csv.dvc", r)
assert r.returncode == 0

with open(os.path.join(CARPETA, "churn.csv.dvc"), encoding="utf-8") as f:
    contenido_recuperado = f.read()
assert contenido_recuperado == contenido_dvc_v1, "el puntero recuperado debería coincidir con el de la v1"

r = ejecutar([DVC, "checkout", "churn.csv.dvc"], cwd=CARPETA)
mostrar("dvc checkout churn.csv.dvc", r)
assert r.returncode == 0

---
## 8. Verificación concreta: ¿es de verdad la v1?

No basta con que los comandos no fallen: leemos el archivo recuperado y comprobamos una estadística real.

In [ ]:
# Verificación concreta: leemos el archivo recuperado y comprobamos una estadística real
df_recuperado = pd.read_csv(os.path.join(CARPETA, "churn.csv"))
maximo_recuperado = int(df_recuperado["tickets_soporte"].max())
suma_recuperada = int(df_recuperado["tickets_soporte"].sum())

print("máximo de tickets_soporte tras dvc checkout:", maximo_recuperado)
print("suma de tickets_soporte tras dvc checkout:", suma_recuperada)

assert maximo_recuperado == maximo_antes, "el máximo recuperado debería ser el de la v1 (sin limitar), no el de la v2"
assert suma_recuperada == suma_antes, "la suma recuperada debería ser la de la v1, no la de la v2"
assert maximo_recuperado != maximo_despues, "si coincidiera con maximo_despues, seguiríamos viendo la v2, no la v1"
print("Confirmado: el archivo recuperado es la v1 original, no la v2 limpiada.")

---
## Resumen de la clase

- Git versiona bien el **código** (el puntero `.dvc`); DVC versiona el **contenido grande** (el dataset o el modelo real), apoyándose en un hash.
- Un **remoto de DVC** puede ser, como en este curso, una simple carpeta local: el mecanismo es el mismo que con un remoto en la nube.
- El **linaje** de una versión combina el commit de Git (qué cambió, cuándo) y el hash de DVC (qué bytes exactos).
- El principio de **datos inmutables** significa que cada limpieza crea una versión nueva, nunca sobrescribe la anterior; la consecuencia directa es más espacio ocupado en el remoto.
- Recuperar una versión anterior requiere **dos pasos** (`git checkout` del puntero + `dvc checkout` del contenido), y conviene verificarlo leyendo el archivo, no solo confiando en que el comando no dio error.

*Continúa con la clase 4: Seguimiento de experimentos.*